## Data Cleaning for corn data

### Corn and ethanol price data

In [225]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

df_corn_price = pd.read_excel("../data/raw-data/corn_ethanol_prices.xlsx")

df_corn_price.columns

Index(['Year', 'Month', 'Corn', 'Ethanol',
       'Blender_cost_of_ethanol_with_credit', 'Gasoline',
       'Corn_cost_per_gallon_of_ethanol', 'Cost_of_ethanol_geg'],
      dtype='object')

In [226]:
# List out fiscal year quarters

quarter_1 = ['Sep', 'Oct', 'Nov']
quarter_2 = ['Dec', 'Jan', 'Feb']
quarter_3 = ['Mar', 'Apr', 'May']
quarter_4 = ['Jun', 'Jul', 'Aug']

df_corn_price["fiscal_quarter"] = None

# Loop through dataframe and match each month to their respective quarter

for i in range(len(df_corn_price)):
    if df_corn_price["Month"][i] in quarter_1:
        df_corn_price.loc[i, "fiscal_quarter"] = "Q1"

    elif df_corn_price["Month"][i] in quarter_2:
        df_corn_price.loc[i, "fiscal_quarter"] = "Q2"

    elif df_corn_price["Month"][i] in quarter_3:
        df_corn_price.loc[i, "fiscal_quarter"] = "Q3"

    elif df_corn_price["Month"][i] in quarter_4:
        df_corn_price.loc[i, "fiscal_quarter"] = "Q4"

# Put year and fiscal quarter columns together into 1 column
df_corn_price["year_quarter"] = df_corn_price["Year"].astype(str) + '_' + df_corn_price["fiscal_quarter"]

In [227]:
df_corn_price_quarter = df_corn_price.groupby("year_quarter").agg({'Corn': 'mean', 'Ethanol': 'mean'}).reset_index()

In [228]:
df_corn_price_quarter = df_corn_price_quarter.rename(columns={"Corn": "corn_price", "Ethanol": "ethanol_price"})

### Corn production split data

In [229]:
df_corn_ethanol_share = pd.read_excel("../data/raw-data/corn_ethanol_share.xlsx")

In [230]:
df_corn_ethanol_share["year_quarter"] = df_corn_ethanol_share["Marketing_year_1"].astype(str) + '_' + df_corn_ethanol_share["Marketing_year_quarter"].str[:2]

In [231]:
# Create percentage share for each category, alcohol is already made

df_corn_ethanol_share["feed_share"] = (df_corn_ethanol_share["Feed_use"] / df_corn_ethanol_share["Total_use"]) *100

df_corn_ethanol_share["food_seed_indust_share"] = (df_corn_ethanol_share["Food_seed_and_industrial_use"] / df_corn_ethanol_share["Total_use"]) *100

df_corn_ethanol_share["exports_share"] = (df_corn_ethanol_share["Exports"] / df_corn_ethanol_share["Total_use"]) * 100

In [232]:
# Create quarterly only data
df_corn_alc_shr_quarterly = df_corn_ethanol_share[df_corn_ethanol_share["year_quarter"].str[5:] != "MY"].reset_index().drop('index', axis=1)

# Create yearly only data
df_corn_alc_shr_yearly = df_corn_ethanol_share[df_corn_ethanol_share["year_quarter"].str[5:] == "MY"].reset_index().drop('index', axis=1)

## Data cleaning for Soybean, poultry, and pork data

In [233]:
# Helper function to change FRED data into same format as corn price data

def to_marketing_quarter(df, value_col, new_col_name):
    df['date'] = pd.to_datetime(df['observation_date'])
    
    def get_mkt_quarter(date):
        month = date.month
        year = date.year
        
        if month in [9, 10, 11]:      # MYQ1
            return f"{year}_Q1"
        elif month in [12, 1, 2]:     # MYQ2
            marketing_year = year if month == 12 else year - 1
            return f"{marketing_year}_Q2"
        elif month in [3, 4, 5]:      # MYQ3
            return f"{year - 1}_Q3"
        else:                          # MYQ4 Jun, Jul, Aug
            return f"{year - 1}_Q4"
    
    df['year_quarter'] = df['date'].apply(get_mkt_quarter)
    df_grouped = df.groupby('year_quarter')[value_col].mean().reset_index()
    df_grouped = df_grouped.rename(columns={value_col: new_col_name})
    
    return df_grouped

### Soybean price data

In [234]:
# Import soybean data

df_soybean_price = pd.read_csv("../data/raw-data/soybean_prices.csv")

df_soybean_price.head()

,observation_date,WPU01830131
0,1947-01-01,51.8
1,1947-02-01,53.5
2,1947-03-01,65.0
3,1947-04-01,62.7
4,1947-05-01,48.7


In [235]:
df_soybean_clean_price = to_marketing_quarter(df_soybean_price, "WPU01830131", "soybean_price")

df_soybean_clean_price.head()

,year_quarter,soybean_price
0,1946_Q2,52.650000
1,1946_Q3,58.800000
2,1946_Q4,53.466667
3,1947_Q1,56.866667
4,1947_Q2,65.066667


### Poultry price data 

In [236]:
df_poultry_price = pd.read_csv("../data/raw-data/poultry_prices.csv")

df_poultry_price.head()

,observation_date,WPS014
0,1967-01-01,56.6
1,1967-02-01,58.5
2,1967-03-01,56.5
3,1967-04-01,57.3
4,1967-05-01,53.6


In [237]:
df_poultry_clean_price = to_marketing_quarter(df_poultry_price, "WPS014", "poultry_price")

df_poultry_clean_price.head()

,year_quarter,poultry_price
0,1966_Q2,57.550000
1,1966_Q3,55.800000
2,1966_Q4,51.066667
3,1967_Q1,46.666667
4,1967_Q2,50.433333


### Pork price data

In [238]:
df_pork_price = pd.read_csv("../data/raw-data/pork_prices.csv")

df_pork_price.head()

,observation_date,WPS022104
0,1974-01-01,67.0
1,1974-02-01,67.4
2,1974-03-01,64.6
3,1974-04-01,61.9
4,1974-05-01,56.2


In [239]:
df_pork_clean_price = to_marketing_quarter(df_pork_price, "WPS022104", "pork_price")

df_pork_clean_price.head()

,year_quarter,pork_price
0,1973_Q2,67.200000
1,1973_Q3,60.900000
2,1973_Q4,60.533333
3,1974_Q1,68.133333
4,1974_Q2,71.433333


## Joining price data

In [240]:
from functools import reduce

dfs = [df_corn_price_quarter, df_poultry_clean_price, df_soybean_clean_price, df_pork_clean_price]
df_merged = reduce(lambda left, right: pd.merge(left, right, on='year_quarter', how='inner'), dfs)

In [241]:
df_merged = df_merged[df_merged['year_quarter'] >= '2000_Q1']
df_merged = df_merged[df_merged['year_quarter'] <= '2015_Q4']

base = df_merged[df_merged['year_quarter'].str.startswith('2005')].mean(numeric_only=True)

# Index all columns a 2005 baseline
df_merged['corn_idx'] = (df_merged['corn_price'] / base['corn_price']) * 100
df_merged['poultry_idx'] = (df_merged['poultry_price'] / base['poultry_price']) * 100
df_merged['soybean_idx'] = (df_merged['soybean_price'] / base['soybean_price']) * 100
df_merged['pork_idx'] = (df_merged['pork_price'] / base['pork_price']) * 100
df_merged['ethanol_idx'] = (df_merged['ethanol_price'] / base['ethanol_price']) * 100

In [243]:
OUT_DIR = os.path.expanduser("../data/processed-data")
os.makedirs(OUT_DIR, exist_ok=True)

df_price_processed = df_merged.copy()
df_price_processed.to_csv(os.path.join(OUT_DIR, "food_prices.csv"), index=False)